In [1]:
#Apply outlier removal and pca
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch
from sklearn.decomposition import PCA

#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

# Drop target and ID column & target column
X = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape:", X.shape)

Features shape: (96000, 7)


In [2]:
# Compute z-scores
z_scores = np.abs((X - X.mean()) / X.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_out = X.drop(index=X.index[outlier_indices])
print("Original features shape:", X.shape)
print("After Outlier Removal:", X_out.shape)


Original features shape: (96000, 7)
After Outlier Removal: (95114, 7)


In [3]:
#apply scaling
scaler = StandardScaler()
X_so = scaler.fit_transform(X_out)

#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_sop = pca.fit_transform(X_so)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_sop.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (95114, 6)
Explained variance ratio sum: 0.9562839882050123


In [4]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [5]:
df_small = df.sample(n=5000, random_state=42)
X_small = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
# Compute z-scores
z_scores = np.abs((X_small - X_small.mean()) / X_small.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_small_clean = X_small.drop(index=X_small.index[outlier_indices])

#apply scaling
X_small_out = scaler.fit_transform(X_small_clean)

#apply pca
X_small_out_pca = pca.fit_transform(X_small_out)

print("Original Outlier Removal:", X_sop.shape)
print("Original Outlier Removal:", X_small_clean.shape)
print("Small dataset Outlier Removal:", X_small_out.shape)
print("Small dataset Outlier Removal:", X_small_out_pca.shape)


Original Outlier Removal: (95114, 6)
Original Outlier Removal: (4961, 7)
Small dataset Outlier Removal: (4961, 7)
Small dataset Outlier Removal: (4961, 6)


In [6]:
#K-Means on Scaled + PCA + outliersremoval Data
start_time = time.time()
kmean_pca_out = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    kmean_pca_out.append({"algorithm": "KMeans", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})


end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"K-means runtime: {runtime:.4f} seconds")  

Runtime: 1119.0730221271515 seconds
K-means runtime: 1119.0730 seconds


In [7]:
#Gaussian Mixture (GMM)on Scaled + PCA + outliersremoval Data
start_time = time.time()
gmm_pca_out = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    gmm_pca_out.append({"algorithm": "GMM", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")    

Runtime: 1383.6151831150055 seconds
GMM runtime: 1383.6152 seconds


In [8]:
#Agglomerative Clustering on Scaled + PCA + outliersremoval Data
start_time = time.time()
agg_pca_out = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_small_out_pca)
    sil, db, ch = compute_metrics(X_small_out_pca, labels)
    agg_pca_out.append({"algorithm": "Agglomerative", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 7.450682640075684 seconds
Agglomerative runtime: 7.4507 seconds


In [9]:
#Spectral Clustering on Scaled + PCA + outliersremoval Data
start_time = time.time()
spec_pca_out = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_small_out_pca)
    sil, db, ch = compute_metrics(X_small_out_pca, labels)
    spec_pca_out.append({"algorithm": "Spectral", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 36.6676230430603 seconds
Spectral runtime: 36.6676 seconds


In [10]:
#DBSCAN on Scaled + PCA + outliersremoval Data
start_time = time.time()
dbscan_pca_out = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_sop)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1 :  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_sop[mask], labels[mask])
        dbscan_pca_out.append({"algorithm": "DBSCAN", "preprocessing": "PCA+Outliers", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 338.2911672592163 seconds
DBSCAN runtime: 338.2912 seconds


In [11]:
#BIRCH on Scaled + PCA + outliersremoval Data
start_time = time.time()
birch_pca_out = []
#threshold_values = [0.2, 0.5, 1.0, 1.5]
threshold_values = [1.5, 3.0, 5.0, 10.0]

for t in threshold_values:
    birch = Birch(n_clusters=None,branching_factor=200, threshold=t)
    labels = birch.fit_predict(X_sop)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_sop) :#and len(set(labels[mask])) > 1
        sil, db, ch = compute_metrics(X_sop, labels)
        birch_pca_out.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA+Outliers",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 150.20033812522888 seconds
BIRCH runtime: 150.2003 seconds


In [12]:
#OPTICS on Scaled + PCA + outliersremoval Data
start_time = time.time()
optics_pca_out = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_sop)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_sop, labels)
        optics_pca_out.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA+Outliers",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"OPTICS runtime: {runtime:.4f} seconds")

Runtime: 11168.544837236404 seconds
OPTICS runtime: 11168.5448 seconds


In [ ]:
import csv


breast_cancer_results_out_pca = (kmean_pca_out + gmm_pca_out + agg_pca_out + spec_pca_out + dbscan_pca_out+birch_pca_out + optics_pca_out)


keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

##with open('updated_data/pain_data/pain_outliers_pca.csv', 'w', newline='') as file:
with open('updated_data/pain_new_data/pain_outliers_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(breast_cancer_results_out_pca)

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

# ARI stability analysis 
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmean_pca_out:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_pca_out:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_pca_out:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spec_pca_out:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_pca_out:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_pca_out:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_pca_out:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# --- helper function to fit a model and return labels ---
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_sop)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_sop), size=len(X_sop), replace=True)
        X_boot = X_sop[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\n---- BOOTSTRAP ARI STABILITY ----")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))

In [ ]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(3).to_string(index=False))

In [ ]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(3).to_string(index=False))

In [ ]:
ari_df.to_csv("updated_data/ARI_Score/pain_pca_outliers_ari.csv", index=False)

In [ ]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmean_pca_out,
    "GMM": gmm_pca_out,
    "Agglomerative": agg_pca_out,
    "Spectral": spec_pca_out,
    "DBSCAN": dbscan_pca_out,
    "BIRCH": birch_pca_out,
    "OPTICS": optics_pca_out
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
    print("="*60)
    print(algorithm)
    print("="*60)



    # Select parameter column


    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break



    # TOP 3 SILHOUETTE


    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )



    # TOP 3 DAVIES-BOULDIN


    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



    # TOP 3 CALINSKI-HARABASZ
  

    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2344
 3      0.1589
 4      0.1477

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.5996
 8          1.7932
 7          1.8538

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         34125.6781
 3         23309.9163
 4         18962.6631


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2310
 3      0.1591
 4      0.1223

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.6145
 3          2.0431
 5          2.2474

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         33289.0861
 3         22732.5133
 4         18042.3762


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1881
 3      0.1192
 4      0.0792

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.8392
 3          2.2579
 8          2.3126

Top 3 Calinski-H

In [ ]:
# Combine all algorithm results 
all_results = (
    kmean_pca_out +
    gmm_pca_out +
    agg_pca_out +
    spec_pca_out +
    dbscan_pca_out +
    birch_pca_out +
    optics_pca_out
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#  Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\nTOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\nTOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better) 
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

#  Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\n BOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Bottom 3 by Davies-Bouldin (higher is worse) 
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\nBOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse) 
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


===== TOP 3 SILHOUETTE =====
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
   DBSCAN NaN  1.0        NaN          NaN         NaN      0.2927
   KMeans 2.0  NaN        NaN          NaN         NaN      0.2344
      GMM 2.0  NaN        NaN          NaN         NaN      0.2310

===== TOP 3 DAVIES-BOULDIN =====
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
   DBSCAN NaN  1.0        NaN          NaN         NaN          0.6652
   DBSCAN NaN  0.5        NaN          NaN         NaN          0.9520
   OPTICS NaN  NaN        NaN          3.0      7802.0          1.2212

===== TOP 3 CALINSKI-HARABASZ =====
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
   KMeans 2.0  NaN        NaN          NaN         NaN         34125.6781
      GMM 2.0  NaN        NaN          NaN         NaN         33289.0861
   KMeans 3.0  NaN        NaN          NaN         NaN         23309.9163

===== BOTTOM 3 SILHOUETTE =====
algorithm   k  eps